In [2]:
## import libraries
import tskit
import numpy as np
import pandas as pd
import os
from typing import Dict, Any, List
from scipy.spatial import distance_matrix
from collections import defaultdict


In [7]:
ts2 = tskit.load('/lisc/data/scratch/admixlab/mk_data/test/test.ts')

ts

In [9]:
import os
import json
import pandas as pd
import numpy as np
import tskit

ts_path = "/lisc/data/scratch/admixlab/mk_data/san/all_chroms_subset.trees"
subset_file = "/lisc/home/user/kuhlwilm/demog-afr/preprocess/Subset_WSS.txt"
out_path = os.path.splitext(ts_path)[0] + "_WSS_subsel.trees"

def pick_name_key(ts, candidates=("variant_data_sample_id", "name", "Name", "sample", "sample_name", "ID", "id")):
    counts = {}
    for key in candidates:
        c = 0
        for ind in ts.individuals():
            md = ind.metadata
            if isinstance(md, dict) and key in md:
                c += 1
        counts[key] = c
    # Pick the key present in the most individuals
    return max(counts.items(), key=lambda kv: kv[1])[0]

def build_name_to_ind_map(ts, key):
    name_to_ind = {}
    for ind_id, ind in enumerate(ts.individuals()):
        md = ind.metadata
        if isinstance(md, dict) and key in md:
            name = str(md[key])
            if name not in name_to_ind:
                name_to_ind[name] = ind_id
    return name_to_ind

def main():
    ts = tskit.load(ts_path)

    # Read the subset file (Pop_old Pop_new Name_old Name_new)
    df = pd.read_csv(subset_file, sep=r"\s+|\t", engine="python")
    for col in ["Pop_old", "Pop_new", "Name_old", "Name_new"]:
        if col not in df.columns:
            raise ValueError(f"Missing column {col} in {subset_file}")
    df = df.assign(
        Pop_old=df["Pop_old"].astype(str).str.strip(),
        Pop_new=df["Pop_new"].astype(str).str.strip(),
        Name_old=df["Name_old"].astype(str).str.strip(),
        Name_new=df["Name_new"].astype(str).str.strip(),
    )

    # Detect which metadata key holds the name; prefer variant_data_sample_id
    name_key = pick_name_key(ts)
    name_to_ind = build_name_to_ind_map(ts, name_key)

    # Collect individuals and sample nodes (supports diploid/multiploid)
    selected_ind_ids = []
    selected_samples = []
    per_ind_sample_counts = []
    for _, row in df.iterrows():
        old_name = row["Name_old"]
        if old_name not in name_to_ind:
            raise ValueError(f"Name_old '{old_name}' not found under metadata key '{name_key}'")
        ind_id = name_to_ind[old_name]
        selected_ind_ids.append(ind_id)
        nodes = ts.individual(ind_id).nodes
        sample_nodes = [n for n in nodes if ts.node(n).is_sample()]
        if len(sample_nodes) == 0:
            raise ValueError(f"Individual '{old_name}' has no sample nodes linked; cannot subset safely.")
        selected_samples.extend(sample_nodes)
        per_ind_sample_counts.append(len(sample_nodes))

    # Simplify to those samples
    ts_sub = ts.simplify(samples=selected_samples, filter_sites=True, filter_individuals=True)

    # Rebuild tables
    tables = ts_sub.dump_tables()
    # Preserve schemas
    tables.populations.metadata_schema = ts.tables.populations.metadata_schema
    tables.individuals.metadata_schema = ts.tables.individuals.metadata_schema

    # Rebuild populations from Pop_new (preserve insertion order)
    tables.populations.clear()
    pop_name_to_id = {}
    for pop_name in pd.unique(df["Pop_new"]):
        tables.populations.add_row(metadata={"name": pop_name})
        pop_name_to_id[pop_name] = len(pop_name_to_id)

    # Map simplified sample nodes back to df rows (simplify keeps sample order as given)
    new_sample_nodes = ts_sub.samples().tolist()
    if len(new_sample_nodes) != sum(per_ind_sample_counts):
        raise AssertionError("Sample node count mismatch after simplify.")
    node_to_ind_row = {}
    idx = 0
    for ind_row_idx, cnt in enumerate(per_ind_sample_counts):
        for _ in range(cnt):
            node_to_ind_row[new_sample_nodes[idx]] = ind_row_idx
            idx += 1

    # Build new individuals in df order, updating metadata
    tables.individuals.clear()
    new_ind_ids = []
    for ind_row_idx, orig_ind_id in enumerate(selected_ind_ids):
        orig_ind = ts.individual(orig_ind_id)
        md = dict(orig_ind.metadata) if isinstance(orig_ind.metadata, dict) else {}
        # Update name and population code
        md[name_key] = df.iloc[ind_row_idx]["Name_new"]
        pop_new = df.iloc[ind_row_idx]["Pop_new"]
        if "group" in md:
            md["group"] = pop_new
        elif "population" in md:
            md["population"] = pop_new
        else:
            md["group"] = pop_new
        # Optionally record numeric population_id in metadata
        md["population_id"] = int(pop_name_to_id[pop_new])

        flags = getattr(orig_ind, "flags", 0)
        location = orig_ind.location if len(orig_ind.location) > 0 else np.array([], dtype=np.float64)

        new_ind_id = tables.individuals.add_row(flags=flags, location=location, metadata=md)
        new_ind_ids.append(new_ind_id)

    # Update nodes: set sample nodes' individual and population
    node_table = tables.nodes
    for s_node in new_sample_nodes:
        ind_row_idx = node_to_ind_row[s_node]
        pop_new = df.iloc[ind_row_idx]["Pop_new"]
        pop_id = pop_name_to_id[pop_new]
        ind_id = new_ind_ids[ind_row_idx]
        row = node_table[s_node]
        node_table[s_node] = row.replace(individual=ind_id, population=pop_id)

    # Build and save
    ts_updated = tables.tree_sequence()
    ts_updated.dump(out_path)
    print(f"Saved updated subset to: {out_path}")
    print(f"Individuals: {ts_updated.num_individuals}, samples: {ts_updated.num_samples}, populations: {ts_updated.num_populations}")

if __name__ == "__main__":
    main()

Saved updated subset to: /lisc/data/scratch/admixlab/mk_data/san/all_chroms_subset_WSS_subsel.trees
Individuals: 8, samples: 16, populations: 2


In [10]:
import tskit
import numpy as np

final_path = "/lisc/data/scratch/admixlab/mk_data/san/all_chroms_subset_WSS_subsel.trees"

def as_text(x):
    # tskit stores states as bytes or strings depending on version/source.
    if x is None:
        return None
    if isinstance(x, bytes):
        try:
            return x.decode("utf-8")
        except Exception:
            # Fallback to repr for non-UTF8
            return repr(x)
    return str(x)

def main():
    ts = tskit.load(final_path)
    tables = ts.tables
    muts = tables.mutations
    sites = tables.sites

    n = muts.num_rows
    if n == 0:
        print("No mutations in the tree sequence.")
        return

    start = max(0, n - 20)
    print(f"Total mutations: {n}. Printing rows [{start}..{n-1}]")

    # Convenience views
    site_pos = sites.position  # numpy array of float64
    # We’ll format positions as integers if they’re integral.
    def fmt_pos(p):
        if abs(p - round(p)) < 1e-9:
            return str(int(round(p)))
        return f"{p:.6f}"

    # Some versions expose arrays for derived_state/ancestral_state only via iteration; we pull per-row
    for i in range(start, n):
        m = muts[i]
        s = sites[m.site]
        pos_str = fmt_pos(site_pos[m.site])

        # Derived state text
        dstate = as_text(m.derived_state)
        astate = as_text(s.ancestral_state)

        # Node/sample flag info (optional)
        node_flags = tables.nodes[m.node].flags
        is_sample = bool(node_flags & tskit.NODE_IS_SAMPLE)

        print(
            f"mut_row={i} site={m.site} pos={pos_str} "
            f"node={m.node}{' (sample)' if is_sample else ''} "
            f"parent={m.parent} time={m.time} "
            f"anc={astate} der={dstate}"
        )

if __name__ == "__main__":
    main()

Total mutations: 1398475. Printing rows [1398455..1398474]
mut_row=1398455 site=1327771 pos=249990609 node=373779 parent=-1 time=52895.59433967701 anc=T der=TGGCAGGG
mut_row=1398456 site=1327772 pos=249990972 node=10 (sample) parent=-1 time=14842.776329311233 anc=T der=C
mut_row=1398457 site=1327773 pos=249991122 node=83026 parent=-1 time=12871.860723028658 anc=G der=A
mut_row=1398458 site=1327774 pos=249991394 node=6 (sample) parent=-1 time=7396.40170355694 anc=A der=C
mut_row=1398459 site=1327775 pos=249991784 node=0 (sample) parent=-1 time=5229.330705233904 anc=G der=T
mut_row=1398460 site=1327776 pos=249992189 node=59653 parent=-1 time=15175.449209630164 anc=T der=C
mut_row=1398461 site=1327777 pos=249992243 node=219156 parent=-1 time=25087.255303319354 anc=C der=T
mut_row=1398462 site=1327778 pos=249992468 node=294218 parent=-1 time=60402.48142653999 anc=G der=A
mut_row=1398463 site=1327779 pos=249992638 node=59653 parent=-1 time=7869.689246389845 anc=G der=A
mut_row=1398464 site=

In [11]:
import tskit
import json
import pprint as pp

ts_path = "/lisc/data/scratch/admixlab/mk_data/san/all_chroms_subset_WSS_subsel.trees"

def try_decode_json(b):
    if b is None:
        return None
    if isinstance(b, (bytes, bytearray)):
        try:
            return json.loads(b.decode("utf-8"))
        except Exception:
            return None
    if isinstance(b, str):
        try:
            return json.loads(b)
        except Exception:
            return None
    return None

def main():
    ts = tskit.load(ts_path)
    print(f"tskit version: {tskit.__version__}")

    # Schemas (if present)
    print("\nPopulation metadata schema:")
    try:
        print(ts.tables.populations.metadata_schema.schema)
    except Exception:
        print("(no schema)")

    print("\nIndividual metadata schema:")
    try:
        print(ts.tables.individuals.metadata_schema.schema)
    except Exception:
        print("(no schema)")

    # Populations: show a couple of examples
    print("\nExample populations (decoded and raw):")
    for i, prow in enumerate(ts.tables.populations):
        if i >= 3:
            break
        decoded = None
        try:
            decoded = ts.tables.populations.metadata_schema.decode_row(prow.metadata)
        except Exception:
            decoded = try_decode_json(prow.metadata)
        print(f"  Pop {i}: decoded={decoded} raw={prow.metadata}")

    # Individuals: show a few examples
    print("\nExample individuals (decoded and raw):")
    for i, irow in enumerate(ts.tables.individuals):
        if i >= 5:
            break
        # High-level object (often already decoded if schema is present)
        decoded_obj = None
        try:
            decoded_obj = ts.individual(i).metadata
        except Exception:
            pass
        # Fallback decode from raw bytes
        if decoded_obj is None or isinstance(decoded_obj, (bytes, bytearray)):
            try:
                decoded_obj = ts.tables.individuals.metadata_schema.decode_row(irow.metadata)
            except Exception:
                j = try_decode_json(irow.metadata)
                if j is not None:
                    decoded_obj = j
        print(f"  Individual {i}:")
        print("    decoded:", decoded_obj)
        print("    raw:", irow.metadata)

    # Optional: show sample node -> individual mapping for first few samples
    print("\nFirst few sample nodes and their individuals:")
    for nid in ts.samples()[:10]:
        node = ts.node(nid)
        print(f"  node {nid}: individual={node.individual}, population={node.population}")

if __name__ == "__main__":
    main()

tskit version: 1.0.3

Population metadata schema:
OrderedDict({'codec': 'json'})

Individual metadata schema:
OrderedDict({'codec': 'json', 'properties': OrderedDict({'variant_data_time': OrderedDict({'default': 0, 'description': 'Time of an individual from the SampleData file.', 'type': 'number'})})})

Example populations (decoded and raw):
  Pop 0: decoded=None raw={'name': 'MSL'}
  Pop 1: decoded=None raw={'name': 'Nama'}

Example individuals (decoded and raw):
  Individual 0:
    decoded: {'variant_data_time': nan, 'group': 'MSL', 'population': 'Mende', 'population_id': 0, 'sex': nan, 'variant_data_sample_id': 'MSL_1'}
    raw: {'variant_data_time': nan, 'group': 'MSL', 'population': 'Mende', 'population_id': 0, 'sex': nan, 'variant_data_sample_id': 'MSL_1'}
  Individual 1:
    decoded: {'variant_data_time': nan, 'group': 'MSL', 'population': 'Mende', 'population_id': 0, 'sex': nan, 'variant_data_sample_id': 'MSL_2'}
    raw: {'variant_data_time': nan, 'group': 'MSL', 'population':